# Stemgedrag & Fractiediscipline

**Thema:** Stemmingen per besluit en de rol van de fracties.

**Kernvragen:** Hoe vaak stemmen fracties unaniem? Zijn er partijen die structureel tegen de meerderheid stemmen, en hoe verhoudt de fractiegrootte zich tot het stemgedrag (vergissingen daargelaten)?

Dit notebook verkent o.a. de silver-tabellen `stemming`, `besluit`, `fractie` en `besluit_zaak`.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True)


def query_df(sql: str, parameters=None) -> pd.DataFrame:
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

## Onderwerp 1

Geef hier een beschrijving van het onderwerp of deelvraag die je wilt beantwoorden.

In [ ]:
for tabel in ["stemming", "besluit", "fractie", "besluit_zaak"]:
    print(f"--- {tabel} ---")
    print(query_df(f"DESCRIBE silver.{tabel}"))
    print()

## Onderwerp 2

...

### Kernvraag 1: Hoe vaak stemmen fracties unaniem?

We bekijken per fractie en per besluit of alle leden hetzelfde stemden (bijvoorbeeld allemaal "Voor"). Vergissingen worden hierbij uitgesloten.

In [ ]:
query = """
WITH fractie_stemmen AS (
    SELECT
        f.naam_nl AS partij,
        s.besluit_id,
        COUNT(DISTINCT s.soort) AS aantal_unieke_stemopties
    FROM silver.stemming s
    JOIN silver.fractie f ON s.fractie_id = f.id
    WHERE s.vergissing = FALSE
    GROUP BY f.naam_nl, s.besluit_id
)
SELECT
    partij,
    COUNT(*) AS totaal_besluiten,
    SUM(CASE WHEN aantal_unieke_stemopties = 1 THEN 1 ELSE 0 END) AS unanieme_besluiten,
    ROUND(100.0 * SUM(CASE WHEN aantal_unieke_stemopties = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS percentage_unaniem
FROM fractie_stemmen
GROUP BY partij
ORDER BY percentage_unaniem DESC
"""
unanimiteit = query_df(query)
unanimiteit

We ontdekten dat bijna elke partij rond de 99-100% unaniem stemt, zelfs grote partijen. Dit controleren we door te kijken hoeveel losse stemrijen er gemiddeld per fractie/besluit zijn:

In [ ]:
query = """
SELECT
    aantal_stemmen_per_fractie_besluit,
    COUNT(*) AS aantal_voorkomens
FROM (
    SELECT besluit_id, fractie_id, COUNT(*) AS aantal_stemmen_per_fractie_besluit
    FROM silver.stemming
    GROUP BY besluit_id, fractie_id
)
GROUP BY aantal_stemmen_per_fractie_besluit
ORDER BY aantal_stemmen_per_fractie_besluit
"""
query_df(query)

**Bevinding:** 932.220 van de ~938.590 fractie/besluit-combinaties (99,3%) hebben maar 1 stemrij. Dit betekent dat bij de meeste besluiten één stem namens de hele fractie wordt geregistreerd (stemming bij handopsteken), in plaats van dat elk Kamerlid apart stemt. Onze eerste analyse is dus grotendeels een artefact van de data. We herhalen de analyse, nu alleen op hoofdelijke stemmingen (waar meerdere losse stemmen per fractie geregistreerd staan).

In [ ]:
query = """
WITH fractie_stemmen AS (
    SELECT
        f.naam_nl AS partij,
        s.besluit_id,
        COUNT(*) AS aantal_stemmen,
        COUNT(DISTINCT s.soort) AS aantal_unieke_stemopties
    FROM silver.stemming s
    JOIN silver.fractie f ON s.fractie_id = f.id
    WHERE s.vergissing = FALSE
    GROUP BY f.naam_nl, s.besluit_id
    HAVING COUNT(*) > 1
)
SELECT
    partij,
    COUNT(*) AS totaal_hoofdelijke_besluiten,
    SUM(CASE WHEN aantal_unieke_stemopties = 1 THEN 1 ELSE 0 END) AS unanieme_besluiten,
    ROUND(100.0 * SUM(CASE WHEN aantal_unieke_stemopties = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS percentage_unaniem
FROM fractie_stemmen
GROUP BY partij
HAVING COUNT(*) >= 5
ORDER BY percentage_unaniem ASC
"""
unanimiteit_hoofdelijk = query_df(query)
unanimiteit_hoofdelijk

In [ ]:
plt.figure(figsize=(10,8))
unanimiteit_hoofdelijk.sort_values("percentage_unaniem").plot.barh(
    x="partij", y="percentage_unaniem",
    title="Percentage unanieme hoofdelijke stemmingen per fractie (min. 5 stemmingen)"
)
plt.xlabel("Percentage unaniem (%)")
plt.tight_layout()
plt.show()

**Bevinding:** bij hoofdelijke stemmingen blijkt de fractiediscipline flink te verschillen. VVD, PvdA en Groep Van Haga stemmen het minst vaak unaniem binnen hun eigen fractie (27-30%), terwijl kleine homogene groepen altijd unaniem stemmen. Dit is een veel eerlijker beeld dan onze eerste analyse.

### Kernvraag 2: Zijn er partijen die structureel tegen de meerderheid stemmen?

Voor elk besluit bepalen we wat de meerderheid stemde (Voor of Tegen), gewogen naar fractiegrootte (zodat een stemming van 10 Kamerleden zwaarder telt dan 1). Daarna checken we per partij hoe vaak ze tegen die meerderheid instemden.

In [ ]:
query = """
WITH fractie_stem AS (
    SELECT
        besluit_id,
        fractie_id,
        soort,
        SUM(COALESCE(fractie_grootte, 1)) AS gewicht,
        ROW_NUMBER() OVER (
            PARTITION BY besluit_id, fractie_id
            ORDER BY SUM(COALESCE(fractie_grootte, 1)) DESC
        ) AS rn
    FROM silver.stemming
    WHERE vergissing = FALSE AND soort IN ('Voor', 'Tegen')
    GROUP BY besluit_id, fractie_id, soort
),
fractie_definitief AS (
    SELECT besluit_id, fractie_id, soort, gewicht
    FROM fractie_stem
    WHERE rn = 1
),
besluit_stemmen AS (
    SELECT besluit_id, soort, SUM(gewicht) AS totaal_gewicht
    FROM fractie_definitief
    GROUP BY besluit_id, soort
),
meerderheid AS (
    SELECT
        besluit_id,
        soort AS meerderheid_soort,
        ROW_NUMBER() OVER (PARTITION BY besluit_id ORDER BY totaal_gewicht DESC) AS rn
    FROM besluit_stemmen
)
SELECT
    f.naam_nl AS partij,
    COUNT(*) AS totaal_besluiten,
    SUM(CASE WHEN fd.soort <> m.meerderheid_soort THEN 1 ELSE 0 END) AS tegen_meerderheid,
    ROUND(100.0 * SUM(CASE WHEN fd.soort <> m.meerderheid_soort THEN 1 ELSE 0 END) / COUNT(*), 1) AS percentage_tegen_meerderheid
FROM fractie_definitief fd
JOIN meerderheid m ON fd.besluit_id = m.besluit_id AND m.rn = 1
JOIN silver.fractie f ON fd.fractie_id = f.id
GROUP BY f.naam_nl
HAVING COUNT(*) >= 20
ORDER BY percentage_tegen_meerderheid DESC
"""
tegen_meerderheid = query_df(query)
tegen_meerderheid

In [ ]:
plt.figure(figsize=(10,10))
tegen_meerderheid.sort_values("percentage_tegen_meerderheid").plot.barh(
    x="partij", y="percentage_tegen_meerderheid",
    title="Percentage besluiten tegen de meerderheid gestemd (per fractie)"
)
plt.xlabel("Percentage tegen meerderheid (%)")
plt.tight_layout()
plt.show()

**Bevinding:** Partij voor de Dieren, Forum voor Democratie en SP stemmen het vaakst tegen de gewogen meerderheid (43-45%) — logisch voor structurele oppositiepartijen. VVD, CDA en ChristenUnie stemmen het vaakst mee met de meerderheid (13-22%), passend bij hun coalitierol.

### Kernvraag 3: hoe verhoudt fractiegrootte zich tot stemgedrag?

We onderzoeken of er een verband is tussen de grootte van een fractie (aantal zetels) en hoe vaak ze tegen de gewogen meerderheid stemmen.

We proberen eerst het aantal zetels op te halen via de kolom `aantal_zetels` in de `fractie`-tabel, gekoppeld aan het aantal besluiten waarop elke fractie heeft gestemd.

In [ ]:
query = """
SELECT
    f.naam_nl AS partij,
    f.aantal_zetels,
    COUNT(*) AS totaal_besluiten
FROM silver.fractie f
JOIN silver.stemming s ON s.fractie_id = f.id
WHERE s.vergissing = FALSE
GROUP BY f.naam_nl, f.aantal_zetels
"""
zetels = query_df(query)
zetels

**Datakwaliteit-opmerking:** de kolom `aantal_zetels` bleek niet bruikbaar — grote, bekende partijen zoals PvdA en GroenLinks stonden op 0 zetels. We vermoeden dat de `fractie`-tabel een aparte rij bevat per periode/samenstelling van een fractie, en verifiëren dit met de actieve/inactieve periodes van deze fracties:

In [ ]:
query = """
SELECT naam_nl, datum_actief, datum_inactief, aantal_zetels
FROM silver.fractie
WHERE naam_nl IN ('Partij van de Arbeid', 'GroenLinks', 'GroenLinks-PvdA', 'Progressief Nederland')
ORDER BY datum_actief
"""
query_df(query)

Dit bevestigt het vermoeden: PvdA en GroenLinks waren actief tot 26-10-2023, waarna ze fuseerden tot "GroenLinks-PvdA" (vanaf 27-10-2023), wat op zijn beurt overging in "Progressief Nederland" vanaf 09-06-2026. De oudere, inactieve fracties tonen daardoor 0 zetels, omdat `aantal_zetels` het meest recente aantal weergeeft, niet het aantal tijdens elke specifieke stemming.

Om deze reden gebruiken we in plaats daarvan de kolom `fractie_grootte` uit de `stemming`-tabel zelf. Deze geeft de fractiegrootte op het moment van elke specifieke stemming, en is daardoor betrouwbaarder voor deze analyse. We berekenen per partij de gemiddelde fractiegrootte over alle stemmingen (met minimaal 100 stemmen, om kleine/kortdurende fracties uit te sluiten).

In [ ]:
query = """
WITH fractie_grootte_gemiddeld AS (
    SELECT
        f.naam_nl AS partij,
        AVG(s.fractie_grootte) AS gemiddelde_fractiegrootte,
        COUNT(*) AS totaal_stemmen
    FROM silver.stemming s
    JOIN silver.fractie f ON s.fractie_id = f.id
    WHERE s.vergissing = FALSE AND s.fractie_grootte IS NOT NULL
    GROUP BY f.naam_nl
    HAVING COUNT(*) >= 100
)
SELECT * FROM fractie_grootte_gemiddeld
ORDER BY gemiddelde_fractiegrootte DESC
"""
fractiegrootte = query_df(query)
fractiegrootte

We combineren deze fractiegrootte-data met de resultaten van kernvraag 2 (percentage tegen de meerderheid gestemd per partij), zodat we beide variabelen naast elkaar kunnen zetten.

In [ ]:
combinatie = fractiegrootte.merge(tegen_meerderheid, on="partij", how="inner")
combinatie

We zetten dit uit in een scatter-plot: op de x-as de gemiddelde fractiegrootte, op de y-as het percentage tegen de meerderheid gestemd. Zo zien we of er een patroon is tussen fractiegrootte en afwijkend stemgedrag.

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(combinatie["gemiddelde_fractiegrootte"], combinatie["percentage_tegen_meerderheid"])

for _, rij in combinatie.iterrows():
    plt.annotate(rij["partij"], (rij["gemiddelde_fractiegrootte"], rij["percentage_tegen_meerderheid"]), fontsize=7)

plt.xlabel("Gemiddelde fractiegrootte (aantal zetels)")
plt.ylabel("Percentage tegen de meerderheid (%)")
plt.title("Verband tussen fractiegrootte en stemgedrag")
plt.tight_layout()
plt.show()

**Bevinding:** er is een duidelijk negatief verband tussen fractiegrootte en het percentage keren dat een fractie tegen de (gewogen) meerderheid stemt. Kleine fracties en eenmansfracties (zoals Ephraim, Bontes, en diverse afsplitsingen) stemmen 35-46% van de tijd tegen de meerderheid, terwijl grote, vaak regerende partijen zoals VVD (13,9%), CDA en Nieuw Sociaal Contract (beide rond de 13%) juist veruit het vaakst met de meerderheid meestemmen.

Dit patroon is goed te verklaren: grote partijen vormen vaker zelf de coalitie en dus de meerderheid (ze stemmen als het ware "met zichzelf mee"), terwijl kleine partijen vaker in de oppositie zitten en zich dus logischerwijs vaker tegen het meerderheidsstandpunt afzetten. Fractiegrootte en coalitie-/oppositierol lijken hier sterk met elkaar samen te hangen, wat ook al naar voren kwam bij kernvraag 2.

### Deelvraag 1: welke partijen staan qua stemgedrag het dichtst bij elkaar?

We berekenen voor elk paar fracties het percentage besluiten waarop ze hetzelfde stemden (Voor/Tegen), en vergelijken dit met hun politieke positionering.

**Stap 1 data ophalen in lang formaat:**

In [ ]:
query = """
WITH fractie_stem AS (
    SELECT
        besluit_id,
        fractie_id,
        soort,
        SUM(COALESCE(fractie_grootte, 1)) AS gewicht,
        ROW_NUMBER() OVER (
            PARTITION BY besluit_id, fractie_id
            ORDER BY SUM(COALESCE(fractie_grootte, 1)) DESC
        ) AS rn
    FROM silver.stemming
    WHERE vergissing = FALSE AND soort IN ('Voor', 'Tegen')
    GROUP BY besluit_id, fractie_id, soort
)
SELECT f.naam_nl AS partij, fs.besluit_id, fs.soort
FROM fractie_stem fs
JOIN silver.fractie f ON fs.fractie_id = f.id
WHERE fs.rn = 1
"""
stemmen_lang = query_df(query)
stemmen_lang.shape

**Stap 2 omzetten naar een partij × besluit tabel, alleen grote/lang-actieve partijen:**

In [ ]:
pivot = stemmen_lang.pivot_table(index="besluit_id", columns="partij", values="soort", aggfunc="first")

# alleen partijen met genoeg stemmen meenemen, anders is de vergelijking onbetrouwbaar
aantal_per_partij = pivot.notna().sum().sort_values(ascending=False)
print(aantal_per_partij)

**Stap 3 bereken de overeenkomst-matrix:**

In [ ]:
grote_partijen = aantal_per_partij[aantal_per_partij >= 20000].index.tolist()
pivot_groot = pivot[grote_partijen]

overeenkomst = pd.DataFrame(index=grote_partijen, columns=grote_partijen, dtype=float)

for p1 in grote_partijen:
    for p2 in grote_partijen:
        if p1 == p2:
            overeenkomst.loc[p1, p2] = 100.0
            continue
        beide_gestemd = pivot_groot[[p1, p2]].dropna()
        if len(beide_gestemd) > 0:
            gelijk = (beide_gestemd[p1].values == beide_gestemd[p2].values).mean()
            overeenkomst.loc[p1, p2] = round(gelijk * 100, 1)

overeenkomst

**Stap 4 visualiseer als heatmap:**

In [ ]:
plt.figure(figsize=(12,10))
plt.imshow(overeenkomst.astype(float), cmap="RdYlGn", vmin=0, vmax=100)
plt.colorbar(label="Percentage overeenkomst (%)")
plt.xticks(range(len(grote_partijen)), grote_partijen, rotation=90)
plt.yticks(range(len(grote_partijen)), grote_partijen)
plt.title("Stemovereenkomst tussen partijen (%)")
plt.tight_layout()
plt.show()

**Stap 5 de meest/minst overeenkomende paren op een rijtje:**

In [ ]:
paren = []
for i, p1 in enumerate(grote_partijen):
    for p2 in grote_partijen[i+1:]:
        paren.append((p1, p2, overeenkomst.loc[p1, p2]))

paren_df = pd.DataFrame(paren, columns=["partij_1", "partij_2", "overeenkomst_pct"]).sort_values("overeenkomst_pct", ascending=False)

print("Meest overeenkomend:")
print(paren_df.head(10))
print("\nMinst overeenkomend:")
print(paren_df.tail(10))

**Bevinding:** de stemovereenkomst tussen partijen volgt grotendeels het verwachte politieke spectrum. Progressief-linkse partijen (GroenLinks, PvdA, Volt, SP, PvdD, DENK) stemmen onderling sterk overeen (83-93%), terwijl partijen van tegenovergestelde zijden van het spectrum (bijv. GroenLinks/PvdD tegenover Forum voor Democratie, of VVD tegenover SP) juist het minst overeenkomen (41-46%). Dit bevestigt dat stemgedrag in de praktijk goed aansluit bij de politieke positionering van partijen — met als kanttekening dat VVD en CDA, traditioneel middenpartijen, relatief hoog scoren met elkaar (82%), wat past bij hun geschiedenis van gezamenlijke coalitiedeelname.

### Deelvraag 2: welke Kamerleden wijken het vaakst af van hun eigen fractie?

We kijken alleen naar hoofdelijke stemmingen (waar leden individueel stemden), bepalen de meerderheidsstem per fractie per besluit, en tellen per Kamerlid hoe vaak ze daarvan afweken.

In [ ]:
query = """
WITH individuele_stemmen AS (
    SELECT
        s.besluit_id,
        s.fractie_id,
        s.persoon_id,
        s.actor_naam,
        s.soort,
        COUNT(*) OVER (PARTITION BY s.besluit_id, s.fractie_id) AS fractie_grootte_stemming
    FROM silver.stemming s
    WHERE s.vergissing = FALSE AND s.soort IN ('Voor', 'Tegen') AND s.persoon_id IS NOT NULL
),
fractie_meerderheid AS (
    SELECT
        besluit_id, fractie_id, soort,
        ROW_NUMBER() OVER (
            PARTITION BY besluit_id, fractie_id
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM individuele_stemmen
    WHERE fractie_grootte_stemming > 1
    GROUP BY besluit_id, fractie_id, soort
),
meerderheid AS (
    SELECT besluit_id, fractie_id, soort AS meerderheid_soort
    FROM fractie_meerderheid WHERE rn = 1
)
SELECT
    f.naam_nl AS partij,
    i.actor_naam AS kamerlid,
    COUNT(*) AS totaal_hoofdelijke_stemmen,
    SUM(CASE WHEN i.soort <> m.meerderheid_soort THEN 1 ELSE 0 END) AS afwijkingen,
    ROUND(100.0 * SUM(CASE WHEN i.soort <> m.meerderheid_soort THEN 1 ELSE 0 END) / COUNT(*), 1) AS percentage_afwijking
FROM individuele_stemmen i
JOIN meerderheid m ON i.besluit_id = m.besluit_id AND i.fractie_id = m.fractie_id
JOIN silver.fractie f ON i.fractie_id = f.id
WHERE i.fractie_grootte_stemming > 1
GROUP BY f.naam_nl, i.actor_naam
HAVING COUNT(*) >= 10
ORDER BY percentage_afwijking DESC
LIMIT 25
"""
afwijkers = query_df(query)
afwijkers

Opvallend: "Boomsma, D.T. (Diederik)" komt voor bij zowel JA21 als Nieuw Sociaal Contract. We controleren of dit een datafout of een politieke overstap is:

In [ ]:
query = """
SELECT DISTINCT i.actor_naam, f.naam_nl AS partij, COUNT(*) AS aantal_stemmen
FROM silver.stemming i
JOIN silver.fractie f ON i.fractie_id = f.id
WHERE i.actor_naam = 'Boomsma, D.T. (Diederik)'
GROUP BY i.actor_naam, f.naam_nl
ORDER BY aantal_stemmen DESC
"""
query_df(query)

**Bevinding:** Kamerleden wijken zelden af van hun fractie (meeste percentages onder de 15%). Diederik Boomsma komt voor bij zowel JA21 (43 stemmen) als Nieuw Sociaal Contract (74 stemmen) dit bleek een politieke overstap tussen fracties, geen datafout. Dit laat zien dat bij dit soort analyses rekening gehouden moet worden met Kamerleden die van fractie wisselen.

### Deelvraag 3: over welk type zaak wordt het vaakst verdeeld gestemd?

We koppelen besluiten via `besluit_zaak` aan de `zaak`-tabel, om te zien of bepaalde soorten zaken (bijv. moties, wetsvoorstellen) vaker tot verdeelde stemmingen leiden dan andere.

stap 1: per besluit bepalen of er verdeeld gestemd is

In [ ]:
query = """
WITH besluit_stemtypes AS (
    SELECT besluit_id, COUNT(DISTINCT soort) AS aantal_stemopties
    FROM silver.stemming
    WHERE vergissing = FALSE AND soort IN ('Voor', 'Tegen')
    GROUP BY besluit_id
),
besluit_met_zaak AS (
    SELECT bz.besluit_id, z.soort AS zaaksoort, bs.aantal_stemopties
    FROM silver.besluit_zaak bz
    JOIN silver.zaak z ON bz.zaak_id = z.id
    JOIN besluit_stemtypes bs ON bz.besluit_id = bs.besluit_id
)
SELECT
    zaaksoort,
    COUNT(*) AS totaal_besluiten,
    SUM(CASE WHEN aantal_stemopties > 1 THEN 1 ELSE 0 END) AS verdeeld_gestemd,
    ROUND(100.0 * SUM(CASE WHEN aantal_stemopties > 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS percentage_verdeeld
FROM besluit_met_zaak
GROUP BY zaaksoort
HAVING COUNT(*) >= 50
ORDER BY percentage_verdeeld DESC
"""
verdeeld_per_zaaksoort = query_df(query)
verdeeld_per_zaaksoort

In [ ]:
plt.figure(figsize=(10, 6))
verdeeld_per_zaaksoort.sort_values("percentage_verdeeld").plot.barh(
    x="zaaksoort", y="percentage_verdeeld",
    title="Percentage verdeeld gestemde besluiten per type zaak",
    legend=False, ax=plt.gca()
)
plt.xlabel("Percentage verdeeld gestemd (%)")
plt.ylabel("")
plt.tight_layout()
plt.show()

**Bevinding:** moties (89,0%) en amendementen (91,0%) worden veruit het vaakst verdeeld gestemd, terwijl uiteindelijke wetgeving (68,3%) en begrotingen (65,2%) juist vaker op bredere steun kunnen rekenen. Dit is goed te verklaren: moties en amendementen zijn vaak bedoeld om een specifiek (politiek) punt te maken en zijn daardoor polariserender van aard, terwijl wetgeving meestal al door een proces van onderhandeling en compromis is gegaan voordat het tot een eindstemming komt.